# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/Users/haikal/Desktop/aie-workspace/AIEC01/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [ ]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

gateway_api_key = getpass("Vercel AI Gateway API Key: ")
os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.16
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [ ]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

# PyPDFLoader extracts one LangChain document per page, including page metadata that survives later chunking
pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

# update each document's metadata
for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

# source_documents is a list of LangChain documents, containing the data & metadata

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/var/folders/l5/sj4yxxdx2858b09x3g1t44qc0000gn/T/ipykernel_40108/1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [7]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [ ]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )

# recall that source_documents is a list of LangChain documents, where each document represents each page from the PDF document. there were 20 pages extracted.

generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [ ]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

# default_transforms_for_prechunked is a default transform pipeline meant for content that's already pre-chunked
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

# for the SummaryExtractor transform, we are augmenting its output to be a NonEmptySummary object
summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    # getattr(...) checks if the transform is a composite/parallel wrapper
    nested = getattr(transform, "transformations", None)

    # this transform is a single, non-composite transform
    if nested is None:
        print(f"- {type(transform).__name__}")

    # this transform is a Parallel, wrapping several transforms that run concurrently
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor: 100%|██████████| 20/20 [00:47<00:00,  2.37s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]/Users/haikal/Desktop/aie-workspace/AIEC01/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.12/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [01:11<00:00,  1.19s/it]
Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 205.05it/s]

KnowledgeGraph(nodes: 20, relationships: 64)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [11]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 64
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [12]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 64)


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

The Ragas graph transforms process was able to enrich the knowledge graph.

Each node was initially `page_content` & `document_metadata`, but now also has `summary`, `summary_embedding`, `theme` & `entities`. Additionally, the graph went from `0` relationships to `64`.

The two relationship types are: `CosineSimilarityBuilder` and `OverlapScoreBuilder`. They are important as they help to build a connected graph, where each node can be connected to one another based on appropriate relationships. This helps to support multi-hop questions, where themes or facts might span across nodes.

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [13]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [14]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

single hop specific
- the query is targeted & the relevant answer is present in a single chunk

multi hop specific
- the query is targeted & the relevant answer is spread out over a few chunks

multi hop abstract
- the query is abstract in itself & the answer isn't directly present in the document, the document will need to be synthesized in order to answer the multi hop abstract query -- the answer is not directly available in the chunks & the query can be more broad or conceptual, thus "abstract"

I expect the multi hop abstract to be the most difficult for a simple RAG application because the LLM doesn't have deep knowledge or understanding of the underlying domain & will rely on looking up the main & relevant keywords from the RAG tool. RAG is basically a search tool & the search query from the LLM is simply matching for terms in the embedding that are semantically similar.

This aside, multi hop abstract queries will likely get missed by a RAG application as only the top `k` chunks get pulled and it's likely that some of the information gets excluded from these `k` chunks as the number of needed chunks grow.

As the multi hop abstract query is "abstract", the keywords or themes in the query may not be explicitly mentioned verbatim in the source document embedded in the RAG & this will be especially where RAG struggles.

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [15]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [00:16<00:00,  2.72s/it]


,user_input,reference,synthesizer_name
0,Who is Paula Plummer mentioned alongside in th...,"Paula Plummer, LVT, VTS (ECC, SAIM), is listed...",single_hop_specific_query_synthesizer
1,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
2,How do the guidelines support individualized l...,The guidelines are intended to support an evol...,multi_hop_abstract_query_synthesizer
3,"According to the themes of ""breed predispositi...",The context says veterinarians should educate ...,multi_hop_abstract_query_synthesizer
4,According to the JAAHA.ORG feline life stage g...,The guidelines state that for cats of all ages...,multi_hop_specific_query_synthesizer
5,What do the American Heartworm Society feline ...,The context says that more detailed informatio...,multi_hop_specific_query_synthesizer


In [16]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

The unrolled generation allows us to inspect every step of the workflow & thus have more observability & insight. We can also cache the enriched graph, as we have done by saving as a `json` file in `artifacts/`.

The rolled generation doesn't allow this & the enriched graph (Knowledhe Graph) is transient & only exists for the duration of that rolled generation call.

Suppose we just want a quick testset generated, we can opt for the rolled workflow.

But if we want to have different testsets, each with different properties (e.g. `query_distribution`, `testset_size`), then the unrolled workflow is better.

An important note is that the entire workflow (from start to finish) is billable, so a consideration is that we can use the cached enriched graph again & not have to pay for the enrichment steps, but just the generation steps.

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [44]:
# Activity #1 workspace
#
# Start with every generated example. Replace this with your reviewed subset.
approved_testset_df = testset_df.copy()
review_status = "review_required"

# Examples:
# approved_testset_df = testset_df.drop(index=[2]).reset_index(drop=True)
# approved_testset_df.loc[0, "user_input"] = "Your clearer question"
# approved_testset_df.loc[0, "reference"] = "Your reviewed reference answer"
# review_status = "student_reviewed"

def print_full_example(df, idx):
    row = df.loc[idx]

    print(f"=== Example {idx}" +  "===" * 50)
    print(f"Synthesizer: {row['synthesizer_name']}")

    print("\nUser Input:")
    print(row["user_input"])

    print("\nReference Contexts:")
    for ctx_idx, context in enumerate(row["reference_contexts"], start=1):
        print(f"  [{ctx_idx}] {context}")

    print("\nReference Answer:")
    print(row["reference"])

    print("\n" + "-" * 80 + "\n")

# reviewing example #1
# print_full_example(approved_testset_df, 0)
'''
Comments:
- Page 1 of the PDF was included in the reference contexts & is relevant to the query
- Despite this, the answer is inadequate. The answer correctly mentions that Paula is one of the authors but doesn't list the other authors as the query was asking for.
  Answer: "Paula Plummer, LVT, VTS (ECC, SAIM), is listed among the authors of the 2021 AAHA/AAFP Feline Life Stage Guidelines."
- Thus, the answer has been updated to more accurately answer the query.
'''
approved_testset_df.loc[0, "reference"] = "Paula Plummer is mentioned alongside other authors Jessica Quimby, Shannon Gowland, Hazel C. Carney, Theresa DePorter, & Jodi Westropp."

# reviewing example #2
# print_full_example(approved_testset_df, 1)
'''
Comments:
- The query is poorly worded, it mentions using the "AAHA" as a means to guide feline healthcare.
  But AAHA stands for American Animal Hospital Association, AAHA is not a tool or mechanism.
  Question: "How do the 2021 AAHA/AAFP Feline Life Stage Guidelines use AAHA to guide feline healthcare throughout a cat’s lifetime?"
- Thus, the user query should be reworded to the following instead: 
  "What framework do the 2021 AAHA/AAFP Feline Life Stage Guidelines provide for guiding feline healthcare throughout a cat's lifetime?"
'''
approved_testset_df.loc[0, "user_input"] = "What framework do the 2021 AAHA/AAFP Feline Life Stage Guidelines provide for guiding feline healthcare throughout a cat's lifetime?"

# reviewing example #3
# print_full_example(approved_testset_df, 2)

'''
User Input:
How do the guidelines support individualized lifelong healthcare, and what individualized care recommendations are made for different feline life stages?

Reference Answer:
The guidelines are intended to support an evolving, individualized, lifelong healthcare strategy for each feline patient at every life stage. They do this by identifying common life-stage features, defining what needs to be done at each stage, and using a prescriptive approach based on a cat’s life stage. They also note that the guidelines are a starting point from which individualized care recommendations can be developed, since some senior cats may remain in excellent condition and be treated as mature adults at the veterinarian’s discretion. For all life stages, the Task Force recommends at least annual examinations, with senior cats seen at least every 6 months and more often if they have chronic conditions. Examples of life-stage-specific recommendations include detecting congenital defects in kittens, preventing obesity in young adults, and increasing vigilance for early renal disease in mature adult and senior cats.

Comments:
- The 2 pulled references were relevant & they showed the 2 distinct sections of the questions.
  The reference answer also answers the query & is able to synthesize the information across the 2 sources.
  No modification is needed for Example #3.
'''

# reviewing example #4
# print_full_example(approved_testset_df, 3)
'''
User Input:
According to the themes of "breed predispositions and genetic risk" and "age-related disease risk in cats," how should a veterinarian tailor preventive care
for a cat across life stages, and what specific guidance does the context give about education, examination frequency, and diagnostics for senior or mature adult cats?

Reference Answer:
The context says veterinarians should educate owners of purebred cats about breed predispositions, while also keeping in mind that most North American cats are
not purebred and that these conditions are not necessarily restricted to particular breeds.

It also explains that age matters because the incidence of many diseases increases as cats age. 

For all cats, the Task Force recommends a minimum of annual examinations, with frequency increased as appropriate for individual needs, and senior
cats should be seen at least every 6 months and more often if they have chronic conditions. For mature adult cats, the Task Force strongly recommends considering T4 testing,
and it notes that regular preventive healthcare examinations and associated medical data can help detect disease early and provide a baseline for later visits. It also states
that comorbidities are extremely common in senior cats and can affect diagnostic, treatment, and management approaches.

Comments:
- The reference answer provided answered the user input accurately & is in-theme with what was asked.
  No modification is required.
'''

# reviewing example #5
# print_full_example(approved_testset_df, 4)

'''
User Input:
According to the JAAHA.ORG feline life stage guidelines, what life stage-based diagnostic and history-taking recommendations are made for cats, 
including when regular fecal examination, heartworm-related testing, and other diagnostics should be considered, and how does team training support implementing these recommendations?

Reference Answer:
The guidelines state that for cats of all ages, the timing and frequency of diagnostics depend on lifestyle, exposure risks, and geographic location.
Regular deworming should be accompanied by fecal examination at intervals based on the patient's health and lifestyle factors.
Heartworm infection is more difficult to diagnose in cats than in dogs because of lower worm burden, single-sex infections, and infrequency of microfilaria,
and HARD adds further complexity; interpretation of antibody and antigen tests is challenging, so a thorough understanding of test limitations is necessary. 
More detailed information is available in the American Heartworm Society guidelines, and testing does not need to be done before starting preventive treatment. 
N-terminal probrain natriuretic peptide has been studied as a diagnostic tool for cardiac disease in cats, but there is limited information to support its use as a screening test, 
so recommendations for general-population frequency cannot be made and use should be individualized with awareness of assay sensitivity and specificity. 

The guidelines also emphasize that practice team training and client education are integral to successful implementation of life stage recommendations, because they help the team 
carry out physical examinations, diagnostics, and treatment protocols appropriately. Feline-specific training should be provided regularly through continuing education, staff meetings, 
and team-building events so that front office staff, technicians, and veterinarians all understand their responsibilities and respond appropriately to the cat's life stage.

Comments:
- Similar to the example #3, this example from the synthesized dataset was able to pull the relevant references
  and the answer provided is accurate & relevant to what was being asked.
  No modification is needed for Example #4
'''

# reviewing example #6
# print_full_example(approved_testset_df, 5)

'''
User Input:
What do the American Heartworm Society feline guidelines say about heartworm testing and prevention in cats?

Reference Answer:
The context says that more detailed information about the limitations of antibody and antigen testing for
heartworm in cats is available in the American Heartworm Society guidelines. 

It also states that testing does not need to be performed before starting preventive treatment.

Comments:
- The 2 references cited are relevant. The first was page 67 of the PDF, second was the citations.
- The reference answer is correct & cites the PDF correctly.
  No modification is required.
'''

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

review_status = "student_reviewed"

,user_input,reference_contexts,reference,synthesizer_name
0,What framework do the 2021 AAHA/AAFP Feline Li...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...,Paula Plummer is mentioned alongside other aut...,single_hop_specific_query_synthesizer
1,How do the 2021 AAHA/AAFP Feline Life Stage Gu...,[Introduction\nThe feline patient ’s life stag...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
2,How do the guidelines support individualized l...,[<1-hop>\n\ntheir cat’s maturation and aging p...,The guidelines are intended to support an evol...,multi_hop_abstract_query_synthesizer
3,"According to the themes of ""breed predispositi...","[<1-hop>\n\nFor example, some senior cats aged...",The context says veterinarians should educate ...,multi_hop_abstract_query_synthesizer
4,According to the JAAHA.ORG feline life stage g...,[<1-hop>\n\ndetection of changes and identi ﬁc...,The guidelines state that for cats of all ages...,multi_hop_specific_query_synthesizer
5,What do the American Heartworm Society feline ...,"[<1-hop>\n\nFor cats of all ages, timing and f...",The context says that more detailed informatio...,multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes

- Example reviewed: all 6 examples were reviewed, but only the first 2 needed modification.
- Decision: the first 2 reference answers were modified.
- Reason: because the reference answers were not correctly answering the user input.
- Any safety or grounding issue found: none, the reference answers were correctly referencing the cited document.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [21]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-fda36d3c
Examples uploaded: 6


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

Your answer here.

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [22]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [23]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [24]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [25]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The retrieved context says a feline wellness visit should consider:

- nutrition and weight management
- oral health
- parasite control
- vaccination
- zoonoses and human safety
- diagnostics
- elimination
- environmental needs

It also notes additional important topics such as:

- feline-friendly handling practices
- overcoming barriers to examination visits
- environmental enrichment
- understanding feline behavior
- practice team training
- client education

The corpus does not provide more detail beyond these listed components.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These recommendations are
---
P

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [26]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [27]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

Your answer here.

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [28]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-1c3010df' at:
https://smith.langchain.com/o/8a61af26-0a1d-42fe-8706-46d289cc4ce1/datasets/2f2036bd-2c9e-4770-9766-3a631010a1f2/compare?selectedSessions=7109673a-1a35-460e-aac8-e5708e44ae44




6it [00:23,  3.98s/it]

Baseline experiment: cat-health-rag-baseline-k3-1c3010df


### Baseline Inspection Notes

- Lowest-scoring example:
- Metric that failed:
- Was the synthetic reference valid?
- Was the retrieved context relevant and sufficient?
- Did the answer add unsupported information?

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [31]:
candidate_retrieval_k = 10
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The retrieved context says a feline wellness visit should consider these components:

- Physical activity and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes that feline-friendly handling practices, overcoming barriers to examination visits, environmental enrichment, understanding feline behavior, practice team training, and client education are important topics to include.

If you want, I can also summarize how the wellness visit focus changes by life stage from the provided context.

Retrieved context count: 10


In [32]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": "deepseek/deepseek-v4-flash",
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-973cb285' at:
https://smith.langchain.com/o/8a61af26-0a1d-42fe-8706-46d289cc4ce1/datasets/2f2036bd-2c9e-4770-9766-3a631010a1f2/compare?selectedSessions=bb773e46-35cc-41d2-bf17-b14ce8692841




6it [00:26,  4.34s/it]

Candidate experiment: cat-health-rag-candidate-k6-973cb285


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

It is a sound approach as it allows us to observe each variable change at a time & its effect on the final result.

The `k` value reflects the number of sources retrieved & thus feeds more data to the LLM. It makes sense that as the value `k` increases, the correctness & groundedness of the answer increases since the model has more context it can reference.

But after the "sweet spot" of `k`, it makes sense that too much sources retrieved could be too much information & it also makes sense that only a % of that `k` sources are relevant & accurate to the premises of the synthetic data generated.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [ ]:
# Activity #2 workspace
#
# A retrieval-depth experiment can reuse the existing vector store:
# student_retrieval_k = 4
# student_target = make_rag_target(student_retrieval_k)
#
# If you change chunking or the embedding model, build a new vector store,
# then create a target with the same output contract:
# {
#     "answer": str,
#     "contexts": list[str],
#     "retrieval_k": int,
# }
#
# Run evaluate(...) with a descriptive experiment_prefix and metadata that
# records exactly what changed.

### 📝 Activity #2 Notes

- Variable changed:
- Prediction:
- Baseline result:
- Candidate result:
- Third experiment result:
- Two traces inspected:
- Decision:
- Cost or latency tradeoff:

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.